In [ ]:
# === SatQuery AI: Pipeline C — Change-Type Segmentation (ResNet34 U-Net) ===
!pip install -q segmentation-models-pytorch huggingface_hub torch torchvision



In [ ]:
import os, json, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import segmentation_models_pytorch as smp
from huggingface_hub import HfApi, login

HF_TOKEN = os.environ.get('HF_TOKEN', '')
if HF_TOKEN:
    login(token=HF_TOKEN)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)



In [ ]:
# === Define 6-Channel ResNet34 U-Net ===
NUM_CLASSES = 5  # 0: no_change, 1: new_construction, 2: demolition, 3: vegetation_growth, 4: deforestation

model = smp.Unet(
    encoder_name='resnet34',
    encoder_weights='imagenet',
    in_channels=6,
    classes=NUM_CLASSES,
    activation=None
).to(device)

print('Model parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))



In [ ]:
# === Synthetic / SECOND Dataset Loader ===
class SyntheticChangeSegDataset(Dataset):
    def __init__(self, num_samples=200, size=256):
        self.num_samples = num_samples
        self.size = size

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        t1 = np.random.uniform(0.1, 0.9, (3, self.size, self.size)).astype(np.float32)
        t2 = t1 + np.random.normal(0, 0.05, (3, self.size, self.size)).astype(np.float32)
        
        mask = np.zeros((self.size, self.size), dtype=np.int64)
        if np.random.rand() > 0.3:
            cx, cy = np.random.randint(50, 200, 2)
            r = np.random.randint(20, 40)
            c_type = np.random.randint(1, 5)
            y, x = np.ogrid[:self.size, :self.size]
            disk = (x - cx)**2 + (y - cy)**2 <= r**2
            mask[disk] = c_type
            t2[:, disk] = np.random.uniform(0.2, 0.8, (3, int(disk.sum())))

        stacked = np.concatenate([t1, t2], axis=0)
        return torch.tensor(stacked), torch.tensor(mask)

dataset = SyntheticChangeSegDataset()
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)



In [ ]:
# === Training Loop ===
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

model.train()
for epoch in range(3):
    total_loss = 0.0
    for imgs, masks in dataloader:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'Epoch {epoch+1}/3 — Loss: {total_loss/len(dataloader):.4f}')

print('Training completed!')



In [ ]:
# === Save Weights & Push to Hugging Face Hub ===
SAVE_DIR = './satquery_ai_change_segmentation'
os.makedirs(SAVE_DIR, exist_ok=True)

weights_file = os.path.join(SAVE_DIR, 'change_segmentation_model.pt')
torch.save(model.state_dict(), weights_file)

config = {
    'classes': NUM_CLASSES,
    'encoder_name': 'resnet34',
    'in_channels': 6,
    'architecture': 'Unet'
}
with open(os.path.join(SAVE_DIR, 'config.json'), 'w') as f:
    json.dump(config, f, indent=2)

HF_REPO = 'mokshda/satquery-ai-change-segmentation'
try:
    api = HfApi()
    api.create_repo(repo_id=HF_REPO, exist_ok=True, private=False)
    api.upload_folder(
        folder_path=SAVE_DIR,
        repo_id=HF_REPO,
        repo_type='model'
    )
    print(f'Successfully uploaded model to https://huggingface.co/{HF_REPO}')
except Exception as e:
    print('Upload error (check HF_TOKEN):', e)

